# Retail Sales Data Processing and Business Insights Generation

## Project Objective

## The objective of this project is to build an end-to-end data engineering pipeline for retail sales data. The workflow includes data ingestion, data cleaning, transformation, validation, KPI generation, privacy masking, and preparation of a curated dataset for Power BI visualization.

## Step 1A: Import Libraries and Load Datasets

## In this step, the required Python libraries are imported and the source datasets are loaded into Pandas DataFrames for further processing.

In [41]:
import pandas as pd

file_path = r'C:\Users\rayin\OneDrive\Documents\Neostats\USECASE - Data Engineering.xlsx'

product_details = pd.read_excel(file_path, sheet_name="product_details")
retail1 = pd.read_excel(file_path, sheet_name="retail_data1")
retail2 = pd.read_excel(file_path, sheet_name="retail_data2")

## Step 1B: Error Handling and Dataset Validation

## This step ensures that the source Excel file is loaded successfully before proceeding with data processing. Basic error handling is implemented to detect issues such as missing files, incorrect file paths, or invalid sheet names. This improves pipeline reliability and prevents failures during downstream transformations.

In [42]:
try:
    file_path = r'C:\Users\rayin\OneDrive\Documents\Neostats\USECASE - Data Engineering.xlsx'

    product_details = pd.read_excel(file_path, sheet_name='product_details')
    retail1 = pd.read_excel(file_path, sheet_name='retail_data1')
    retail2 = pd.read_excel(file_path, sheet_name='retail_data2')

    print("All datasets loaded successfully.")
    print(f"Product Details Records: {product_details.shape[0]}")
    print(f"Retail Data 1 Records: {retail1.shape[0]}")
    print(f"Retail Data 2 Records: {retail2.shape[0]}")

except FileNotFoundError:
    print("Error: Excel file not found. Please verify the file path.")

except ValueError as e:
    print(f"Error: Sheet name not found. {e}")

except Exception as e:
    print(f"Unexpected error occurred: {e}")

All datasets loaded successfully.
Product Details Records: 10
Retail Data 1 Records: 4243
Retail Data 2 Records: 4251


## Step 2: Data Understanding and Initial Assessment

## This step examines the structure of the datasets, including column names, record counts, data types, missing values, and duplicate records.

In [43]:
print(product_details.head())
print(retail1.head())
print(retail2.head())

   product_id product_name     category   price
0         101       Laptop  Electronics  250000
1         102        Phone  Electronics   70000
2         103        Shirt     Clothing    1300
3         104        Shoes     Clothing    8000
4         105           TV  Electronics   45000
   transaction_id  customer_id   customer_name  product_id    price  \
0               1          642   Troy Mitchell         108  15000.0   
1               2          881  Sarah Guerrero         102  70000.0   
2               3          505   Samantha Hull         109  65000.0   
3               4          794   Gerald Cooper         106  80000.0   
4               5          864   Cameron Black         108  15000.0   

    product_name         category purchase_location       city  \
0  Mixer grinder  Home Appliances           offline  Bangalore   
1          Phone             ELEC            online  Bangalore   
2   Refrigerator  Home Appliances           offline    Chennai   
3           sofa     

In [44]:
print(product_details.shape)
print(retail1.shape)
print(retail2.shape)

(10, 4)
(4243, 16)
(4251, 16)


In [45]:
print(retail1.columns)

Index(['transaction_id', 'customer_id', 'customer_name', 'product_id', 'price',
       'product_name', 'category', 'purchase_location', 'city',
       'transaction_date', 'quantity', 'payment_method', 'discount', 'email',
       'phone', 'payment_status'],
      dtype='object')


In [46]:
print(retail1.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4243 entries, 0 to 4242
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     4243 non-null   int64  
 1   customer_id        4243 non-null   int64  
 2   customer_name      4243 non-null   object 
 3   product_id         4243 non-null   int64  
 4   price              3839 non-null   float64
 5   product_name       4243 non-null   object 
 6   category           4243 non-null   object 
 7   purchase_location  4243 non-null   object 
 8   city               4243 non-null   object 
 9   transaction_date   4243 non-null   object 
 10  quantity           4243 non-null   int64  
 11  payment_method     4243 non-null   object 
 12  discount           4243 non-null   float64
 13  email              4243 non-null   object 
 14  phone              4243 non-null   int64  
 15  payment_status     4243 non-null   object 
dtypes: float64(2), int64(5),

In [47]:
print("Retail 1 Null Data\n")
print(retail1.isnull().sum())
print("\n Retail 2 Null Data\n")
print(retail2.isnull().sum())

Retail 1 Null Data

transaction_id         0
customer_id            0
customer_name          0
product_id             0
price                404
product_name           0
category               0
purchase_location      0
city                   0
transaction_date       0
quantity               0
payment_method         0
discount               0
email                  0
phone                  0
payment_status         0
dtype: int64

 Retail 2 Null Data

transaction_id         0
customer_id            0
customer_name          0
product_id             0
price                405
product_name           0
category               0
purchase_location      0
city                   0
transaction_date       0
quantity               0
payment_method         0
discount               0
email                  0
phone                  0
payment_status         0
dtype: int64


In [48]:
print(retail1.duplicated().sum())
print(retail2.duplicated().sum())

0
0


## Step 3: Dataset Consolidation

## The retail transaction datasets are combined into a single dataset to create a unified source for data cleaning and analysis.

In [49]:
retail_data = pd.concat([retail1, retail2], ignore_index=True)

In [50]:
retail_data.shape

(8494, 16)

## Step 4: Duplicate Detection and Removal

## Duplicate transaction records are identified and removed to prevent inaccurate reporting and revenue inflation.

In [51]:
retail_data.drop_duplicates(inplace=True)

In [52]:
retail_data.duplicated().sum()

0

## Step 5: Missing Value Handling 

## Missing product prices are identified and populated using the Product Details reference dataset.

In [53]:
retail_data['price'].isnull().sum()

809

In [54]:
price_map = product_details.set_index('product_id')['price']

retail_data['price'] = retail_data['price'].fillna(
    retail_data['product_id'].map(price_map)
)

In [55]:
retail_data['price'].isnull().sum()

0

## Step 6: Product Name Standardization  

## Product names are standardized to ensure consistency across all records and improve reporting accuracy.

In [56]:
retail_data['product_name'] = retail_data['product_name'].str.title()

## Step 7: Category Standardization 

## Category names are cleaned and standardized into business-friendly formats.

In [57]:
category_map = {
    'ELEC':'Electronics',
    'electronics':'Electronics',
    'FURN':'Furniture',
    'furniture':'Furniture',
    'HOME':'Home Appliances',
    'home appliances':'Home Appliances',
    'CLOTH':'Clothing',
    'clothing':'Clothing'
}

retail_data['category'] = retail_data['category'].replace(category_map)

In [58]:
print(retail_data['category'].unique())

['Home Appliances' 'Electronics' 'Furniture' 'Clothing']


## Step 8: Date Standardization 
## Transaction dates are converted into a consistent format suitable for reporting and trend analysis.

In [59]:
def fix_date(date_str):

    try:
        parts = str(date_str).split('-')
        first = int(parts[0])
        second = int(parts[1])

        if first > 12:
            return pd.to_datetime(date_str,format='%d-%m-%Y')
        elif second > 12:
            return pd.to_datetime(date_str,format='%m-%d-%Y')
        else:
            return pd.to_datetime(date_str,format='%d-%m-%Y')

    except:
        return pd.NaT

In [60]:
retail_data['transaction_date'] = retail_data['transaction_date'].apply(fix_date)
retail_data['transaction_date'] = retail_data['transaction_date'].dt.strftime('%d-%m-%Y')

## Step 9: City Name Standardization 

## City names are standardized to eliminate formatting inconsistencies.

In [61]:
retail_data['city'] = retail_data['city'].str.title()

## Step 10: Purchase Location Standardization 

## Purchase location values are cleaned and standardized to maintain consistency.

In [62]:
retail_data['purchase_location'] = retail_data['purchase_location'].str.capitalize()

## Step 11: Personally Identifiable Information (PII) Masking

## Sensitive customer information such as email addresses and phone numbers is masked to protect privacy.

In [63]:
def mask_email(email):
    email = str(email)
    username, domain = email.split('@')
    return username[:2] + '****@' + domain

retail_data['email_masked'] = retail_data['email'].apply(mask_email)

In [64]:
def mask_phone(phone):
    phone = str(phone)
    return '*'*(len(phone)-4) + phone[-4:]

retail_data['phone_masked'] = retail_data['phone'].apply(mask_phone)

## Step 12: Revenue Calculation

## A Revenue field is generated using the formula: Revenue = Price × Quantity × (1 − Discount). This field serves as the primary business metric used throughout the dashboards.

In [65]:
retail_data['Revenue'] = (retail_data['price']* retail_data['quantity'] * (1 - retail_data['discount']))

## Step 13: KPI Generation

## Key performance indicators are calculated to understand overall business performance and support dashboard development.

In [66]:
total_revenue = retail_data['Revenue'].sum()
print(total_revenue)

1233270150.0


In [67]:
revenue_category = retail_data.groupby('category')['Revenue'].sum().reset_index()

In [68]:
revenue_city = retail_data.groupby('city')['Revenue'].sum().reset_index()

In [69]:
top_products = retail_data.groupby('product_name')['Revenue'].sum().sort_values(ascending=False).head(10)

## Step 14: Export Cleaned Dataset 

## Final cleaned and transformed dataset is exported to Excel and used as the source for Power BI reporting.

In [70]:
retail_data.to_excel("cleaned_retail_data.xlsx",index=False)

## Pipeline Summary:

## The data pipeline successfully performed:
## - Data ingestion
## - Data validation
## - Missing value handling
## - Duplicate removal
## - Data standardization
## - PII masking
## - Revenue calculation
## - KPI generation
## - Export of analytics-ready dataset
## The final cleaned dataset was used as the source for Power BI dashboard development.

# Conclusion:

## The raw retail sales data was successfully cleaned, standardized, transformed, and enriched with business metrics.
## The final curated dataset was prepared for Power BI visualization and business insight generation.